In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly import graph_objects as go
from datetime import datetime
import pycountry
import numpy as np
import h5py  # Make sure h5py initialises properly before pandas ruins it

from emu_renewal.constants import ANALYSIS_TYPES, BASE_PATH, G_MOB_LOCATION_CMAP, OXCGRT_LOCATION_CMAP, MOBILITY_SMOOTH_PERIOD
from emu_renewal.inputs import DATA_PATH, get_oxcgrt
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_analysis_fit, plot_prior_post, plot_imm_props, get_google_mobility

In [ ]:
iso3 = "COL"
analysis = "oxcgrt"
run_name = "20251121_1236"
pycountry.countries.lookup(iso3)

In [ ]:
path = BASE_PATH / "outputs" / run_name / iso3 / analysis
spaghetti = pd.read_hdf(path / "spaghetti.h5")
targets = load_targets(path)
idata = az.from_netcdf(path / "idata_filtered.nc")

In [ ]:
outputs = list(targets.keys()) + ["process"]
spagh_plot = plot_analysis_fit(spaghetti, targets, outputs)

if analysis == "g_mob":
    scaling_series = get_google_mobility(iso3)
    loc_map = G_MOB_LOCATION_CMAP
else:
    scaling_series = get_oxcgrt(iso3, "custom")
    loc_map = OXCGRT_LOCATION_CMAP
smoothed_scaling = scaling_series.rolling(MOBILITY_SMOOTH_PERIOD, center=True).mean().dropna()
filtered_scaling = smoothed_scaling.loc[smoothed_scaling.index < spaghetti.index[-1]]
for l in filtered_scaling.columns:
    colour = loc_map[l]
    spagh_plot.add_traces(go.Scatter(x=filtered_scaling.index, y=filtered_scaling[l], line={"color": colour}, name=l), rows=outputs.index("process") + 1, cols=1)

In [ ]:
spagh_plot

In [ ]:
plot_imm_props(spaghetti)

In [ ]:
priors = pickle.load(open(path / "priors.pkl", "rb"))
epi_params = [p for p in priors if p != "shared_dispersion"]
plot_prior_post(iso3, epi_params, priors, idata);